# PolarisT atlas-unprofiled driver discovery demo

This notebook demonstrates how to define a desired CD8+ T-cell state and use the pretrained PolarisT model to rank atlas-unprofiled candidate genes.

Complete [package installation](../README.md#1-package-installation) before running this notebook. Then follow the steps below.


## Data preparation

Download [Anndata_cd8_raw.h5ad](https://figshare.com/ndownloader/files/66953270) and save it in a local data directory. The dataset is documented in the associated [Figshare record](https://doi.org/10.6084/m9.figshare.32934569).

Set `resource_dir` below to your actual data directory. For example, if the file is saved as `/path/to/polarist_data/Anndata_cd8_raw.h5ad`, use `/path/to/polarist_data`.

**Note:** `resource_dir` must point to the directory containing the file, not to the `.h5ad` file itself.


In [ ]:
# Replace with the directory containing Anndata_cd8_raw.h5ad.
resource_dir = "/path/to/polarist_data"


In [ ]:
from polarist import rank_unseen_drivers

## Input parameters

- `phenotype_name`: phenotype label used in output filenames.
- `positive_genes`: genes expected to be highly expressed in the desired state.
- `negative_genes`: genes expected to be weakly expressed in the desired state; optional.
- `extreme_fraction`: fraction selected from each expression-score tail within every dataset. Valid range: `(0, 0.5]`. Default: `0.05`.
- `n_genes`: number of top-ranked atlas-unprofiled genes returned. Accepted values: `100`, `1000`, or `None` for all candidates. Default: `1000`.
- `tf_only`: whether to additionally generate the transcription-factor subset in `result.tf_ranking`.
- `resource_dir`: local directory containing the downloaded `Anndata_cd8_raw.h5ad` file.


## Define the desired phenotype

This example defines a CD8+ T-cell objective with increased stemness and reduced exhaustion. Stemness-associated genes form the positive signature, while exhaustion-associated genes form the negative signature.

To define a custom phenotype, replace `positive_genes` and `negative_genes` and update `phenotype_name`. Use the same `resource_dir` set above. `negative_genes` can be omitted when using a positive signature alone.


In [ ]:
phenotype_name = "stemness"
extreme_fraction = 0.05
n_genes = 1000

positive_genes = [
    "TCF7", "LEF1", "SLAMF6", "SELL", "BCL2",
    "BCL6", "CXCR5", "CCNE1", "CCNE2",
]
negative_genes = ["TOX", "HAVCR2", "ENTPD1", "CD101", "CD244"]


## Rank atlas-unprofiled genes


In [ ]:
result = rank_unseen_drivers(
    positive_genes=positive_genes,
    negative_genes=negative_genes,
    phenotype_name=phenotype_name,
    extreme_fraction=extreme_fraction,
    n_genes=n_genes,
    tf_only=True,
    resource_dir=resource_dir,
)

## Full ranking

Top atlas-unprofiled candidate genes ranked toward the desired phenotype.

In [ ]:
result.full_ranking.head(20)

## Transcription-factor ranking

In [ ]:
result.tf_ranking.head(20)

## Save the rankings

The output filenames are generated automatically from `phenotype_name`.

In [ ]:
result.full_ranking.to_csv(
    f"atlas-unprofiled_{result.phenotype_name}_full_ranking.csv",
    index_label="Perturbation",
)
result.tf_ranking.to_csv(
    f"atlas-unprofiled_{result.phenotype_name}_tf_ranking.csv",
    index_label="Perturbation",
)